## GPS Consumer

In [0]:
# Configuration
# Creating widgets
dbutils.widgets.text("storage_account", "dlspl21databricks", "Storage account")
dbutils.widgets.text("container", "live-transit-monitor", "Container")
dbutils.widgets.text("catalog", "dbr_dev", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "live_transit_monitor", "Bronze schema")

# Reading the values from the widgets
STORAGE   = dbutils.widgets.get("storage_account")
CONTAINER = dbutils.widgets.get("container")
CATALOG   = dbutils.widgets.get("catalog")
SCHEMA    = dbutils.widgets.get("bronze_schema")

BASE            = f"abfss://{CONTAINER}@{STORAGE}.dfs.core.windows.net"
CHECKPOINT      = f"{BASE}/_checkpoint/gps_data"
BRONZE_GPS_DATA = f"{CATALOG}.{SCHEMA}.gps_data"

In [0]:
# Checking the values
print("CONTAINER =", CONTAINER)
print("CHECKPOINT =", CHECKPOINT)
print("TABLE =", BRONZE_GPS_DATA)

In [0]:
# Event Hub (Kafka endpoint)
dbutils.widgets.text("eh_namespace", "evhpl24databricks02", "EH namespace")
dbutils.widgets.text("eh_name", "live_transit_monitor_evh", "EH name (hub/topic)")
dbutils.widgets.text("consumer_group", "$Default", "Consumer group")
dbutils.widgets.text("secret_scope", "default2", "Secret scope")
dbutils.widgets.text("secret_key", "eh-conn-transit", "Secret key (Listen)")
dbutils.widgets.text("starting_offsets", "earliest", "Starting offsets")

EH_NAMESPACE = dbutils.widgets.get("eh_namespace")
EH_NAME      = dbutils.widgets.get("eh_name")
CONSUMER     = dbutils.widgets.get("consumer_group")
STARTING     = dbutils.widgets.get("starting_offsets")

BOOTSTRAP = f"{EH_NAMESPACE}.servicebus.windows.net:9093"
EH_CONN   = dbutils.secrets.get(dbutils.widgets.get("secret_scope"),
                                dbutils.widgets.get("secret_key"))   
EH_JAAS   = ("kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
            f'username="$ConnectionString" password="{EH_CONN}";')



In [0]:
# Reading the data from Event Hub
raw = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", EH_NAME)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", EH_JAAS)
    .option("startingOffsets", STARTING)
    .option("failOnDataLoss", "false")    # Reading from the earliest available data
    .load())



In [0]:
from pyspark.sql.types import StructType, StringType, IntegerType, FloatType
from pyspark.sql.functions import from_json, col, to_timestamp, current_timestamp, lit

# Schema - identical with raw JSON file
event_schema = (StructType()
    .add("generated", StringType())
    .add("routeShortName", StringType())
    .add("tripId", IntegerType())
    .add("routeId", IntegerType())
    .add("headsign", StringType())
    .add("vehicleCode", StringType())
    .add("vehicleService", StringType())
    .add("vehicleId", IntegerType())
    .add("speed", IntegerType())
    .add("direction", IntegerType())
    .add("delay", IntegerType())
    .add("scheduledTripStartTime", StringType())
    .add("lat", FloatType())
    .add("lon", FloatType())
    .add("gpsQuality", IntegerType())
    .add("lastUpdate", StringType())
)

parsed = (raw
    .select(
        from_json(col("value").cast("string"), event_schema).alias("e"),  # binary -> string -> JSON
        col("partition"), col("offset"),
        col("timestamp").alias("enqueued_ts"))          # when it landed in Event Hub
    .select("e.*", "partition", "offset", "enqueued_ts")
    .withColumn("event_time", to_timestamp("generated"))  # event-time from source
    .withColumn("_source", lit("eventhub"))
    .withColumn("ingestion_ts", current_timestamp()))     # when we ingested


In [0]:
%sql

CREATE EXTERNAL LOCATION IF NOT EXISTS live_transit_monitor
  URL 'abfss://live-transit-monitor@dlspl21databricks.dfs.core.windows.net/'
  WITH (STORAGE CREDENTIAL `databricks_uc_connector`);

In [0]:
query = (parsed.writeStream
   .format("delta")
   .option("checkpointLocation", CHECKPOINT)
   .option("mergeSchema", "true")
   .trigger(availableNow=True)
   .toTable(BRONZE_GPS_DATA))

query.awaitTermination()  

In [0]:
# Checking if the data was written -- comment it out for production
display(spark.read.table(BRONZE_GPS_DATA).orderBy(col("ingestion_ts").desc()).limit(20))
spark.read.table(BRONZE_GPS_DATA).count()